## Configuration Contracts

In [2]:

from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple, Any
from enum import Enum
from pathlib import Path
import json

@dataclass
class VertexAIConfig:
    """Configuration for Vertex AI services."""
    project_id: str
    location: str
    bucket_name: str
    credentials_path: str = "vertex-ai-key.json"
    staging_bucket: Optional[str] = None
    
    def __post_init__(self):
        if self.staging_bucket is None:
            self.staging_bucket = f"gs://{self.bucket_name}/staging"

@dataclass
class TrainingJobConfig:
    """Configuration for a training job."""
    display_name: str
    model_name: str = "cardiffnlp/twitter-roberta-base-sentiment-latest"
    learning_rate: float = 2e-5
    batch_size: int = 32
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    num_epochs: int = 4
    max_length: int = 128
    machine_type: str = "n1-standard-4"
    accelerator_type: str = "NVIDIA_TESLA_T4"
    accelerator_count: int = 1

@dataclass
class HyperparameterTuningConfig:
    """Configuration for hyperparameter tuning."""
    display_name: str
    max_trial_count: int = 10
    parallel_trial_count: int = 2
    metric_id: str = "f1_macro"
    metric_goal: str = "maximize"
    search_space: Dict[str, Dict[str, Any]] = field(default_factory=dict)
    
    def __post_init__(self):
        if not self.search_space:
            self.search_space = {
                "learning_rate": {"type": "double", "min": 5e-6, "max": 5e-5, "scale": "log"},
                "batch_size": {"type": "discrete", "values": [16, 32]},
                "weight_decay": {"type": "double", "min": 0.001, "max": 0.1, "scale": "linear"},
                "warmup_ratio": {"type": "double", "min": 0.0, "max": 0.3, "scale": "linear"}
            }

class ModelType(Enum):
    """Supported model types."""
    ROBERTA_TWITTER = "cardiffnlp/twitter-roberta-base-sentiment-latest"
    BERT_BASE = "bert-base-uncased"
    DISTILBERT = "distilbert-base-uncased"

print("[SUCCESS] Configuration contracts defined")

[SUCCESS] Configuration contracts defined


## Initialization API
# 
# Initialize Vertex AI with credentials and project settings.

In [5]:
import vertex_ai_utils as vai
from google.cloud import aiplatform

def initialize_vertex_ai(config: VertexAIConfig) -> None:
    """
    Initialize Vertex AI SDK.
    
    Args:
        config: VertexAIConfig instance
    """
    vai.initialize_vertex_ai(
        project_id=config.project_id,
        location=config.location,
        credentials_path=config.credentials_path,
        staging_bucket=config.staging_bucket
    )
    print(f"[SUCCESS] Vertex AI initialized for project: {config.project_id}")

/Users/adwaith/Desktop/AML/aml/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


An error occurred: module 'importlib.metadata' has no attribute 'packages_distributions'


/Users/adwaith/Desktop/AML/aml/lib/python3.9/site-packages/google/api_core/_python_version_support.py:252: FutureWarning: You are using a Python version (3.9.6) past its end of life. Google will update google.api_core with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)


# Data Upload API 
# Upload datasets to Google Cloud Storage.

In [6]:
def upload_dataset(
    config: VertexAIConfig,
    train_path: str,
    val_path: str,
    test_path: str,
    destination_folder: str = "sentiment-data"
) -> Tuple[str, str, str]:
    """
    Upload train/val/test datasets to GCS.
    
    Args:
        config: VertexAIConfig instance
        train_path: Local path to training data (JSONL)
        val_path: Local path to validation data (JSONL)
        test_path: Local path to test data (JSONL)
        destination_folder: GCS folder name
        
    Returns:
        Tuple of (train_uri, val_uri, test_uri)
    """
    print(f"[UPLOAD] Uploading datasets to gs://{config.bucket_name}/{destination_folder}/")
    
    train_uri = vai.upload_to_gcs(
        bucket_name=config.bucket_name,
        source_file_path=train_path,
        destination_blob_name=f"{destination_folder}/train.jsonl"
    )
    
    val_uri = vai.upload_to_gcs(
        bucket_name=config.bucket_name,
        source_file_path=val_path,
        destination_blob_name=f"{destination_folder}/val.jsonl"
    )
    
    test_uri = vai.upload_to_gcs(
        bucket_name=config.bucket_name,
        source_file_path=test_path,
        destination_blob_name=f"{destination_folder}/test.jsonl"
    )
    
    print(f"\n[SUCCESS] All datasets uploaded!")
    return train_uri, val_uri, test_uri

## Create and run training jobs.


In [ ]:
def create_roberta_training_job(
    config: VertexAIConfig,
    job_config: TrainingJobConfig,
    train_data_uri: str,
    val_data_uri: str,
    test_data_uri: str,
    training_script_path: str = "vertex_ai_training.py",
    metric_name: str = "f1_macro"
) -> aiplatform.CustomJob:
    """
    Create a RoBERTa training job.
    
    Args:
        config: VertexAIConfig instance
        job_config: TrainingJobConfig with hyperparameters
        train_data_uri: GCS URI to training data
        val_data_uri: GCS URI to validation data
        test_data_uri: GCS URI to test data
        training_script_path: Path to training script
        
    Returns:
        CustomJob instance (not yet started)
    """
    print(f"[INFO] Creating RoBERTa training job: {job_config.display_name}")
    
    job = vai.create_custom_roberta_training_job(
        display_name=job_config.display_name,
        script_path=training_script_path,
        train_data_gcs_uri=train_data_uri,
        val_data_gcs_uri=val_data_uri,
        test_data_gcs_uri=test_data_uri,
        base_output_dir=f"gs://{config.bucket_name}/model-output",
        project_id=config.project_id,
        location=config.location,
        learning_rate=job_config.learning_rate,
        batch_size=job_config.batch_size,
        weight_decay=job_config.weight_decay,
        warmup_ratio=job_config.warmup_ratio,
        num_epochs=job_config.num_epochs,
        metric_name=metric_name
    )
    
    print(f"[SUCCESS] Training job created: {job_config.display_name}")
    return job

def run_training_job(
    job: aiplatform.CustomJob,
    sync: bool = True
) -> aiplatform.CustomJob:
    """
    Run a training job.
    
    Args:
        job: CustomJob instance
        sync: If True, wait for completion
        
    Returns:
        CustomJob instance (completed if sync=True)
    """
    print(f"[START]  Starting training job...")
    
    if sync:
        print(f"   [WAIT] Running synchronously (will wait for completion)...")
        print(f"   Monitor: https://console.cloud.google.com/vertex-ai/training/custom-jobs")
    else:
        print(f"   [ASYNC] Running asynchronously (will return immediately)")
    
    # Run the job using the SDK method directly
    #job.run(sync=sync)
    print("Running job with service account...")
    job.run(service_account='vertex-ai-service-account@noted-cortex-477800-b7.iam.gserviceaccount.com', sync=sync)
    if sync:
        print(f"\n[SUCCESS] Training job completed!")
        print(f"   Job name: {job.display_name}")
        print(f"   Job state: {job.state}")
    else:
        print(f"\n[SUCCESS] Training job submitted!")
        print(f"   Use job.wait() to wait for completion")
    
    return job


BERT Baseline API (Bonus)

In [ ]:
def create_bert_training_job(
    config: VertexAIConfig,
    job_config: TrainingJobConfig,
    train_data_uri: str,
    val_data_uri: str,
    test_data_uri: str,
    training_script_path: str = "vertex_ai_training.py",
    metric_name: str = "f1_macro"
) -> aiplatform.CustomJob:
    """
    Create a BERT baseline training job.
    
    Args:
        config: VertexAIConfig instance
        job_config: TrainingJobConfig with hyperparameters
        train_data_uri: GCS URI to training data
        val_data_uri: GCS URI to validation data
        test_data_uri: GCS URI to test data
        training_script_path: Path to training script
        
    Returns:
        CustomJob instance (not yet started)
    """
    print(f"[INFO] Creating BERT baseline job: {job_config.display_name}")
    print(f"[INFO] BONUS: For transfer learning comparison")
    
    job = vai.create_custom_bert_training_job(
        display_name=job_config.display_name,
        script_path=training_script_path,
        train_data_gcs_uri=train_data_uri,
        val_data_gcs_uri=val_data_uri,
        test_data_gcs_uri=test_data_uri,
        base_output_dir=f"gs://{config.bucket_name}/model-output",
        project_id=config.project_id,
        location=config.location,
        learning_rate=job_config.learning_rate,
        batch_size=job_config.batch_size,
        weight_decay=job_config.weight_decay,
        warmup_ratio=job_config.warmup_ratio,
        num_epochs=job_config.num_epochs,
        metric_name=metric_name
    )
    
    print(f"[SUCCESS] BERT training job created: {job_config.display_name}")
    return job

def run_bert_training_job(
    job: aiplatform.CustomJob,
    sync: bool = True
) -> aiplatform.CustomJob:
    """
    Run a BERT training job.
    
    Args:
        job: CustomJob instance
        sync: If True, wait for completion
        
    Returns:
        CustomJob instance (completed if sync=True)
    """
    print(f"[START]  Starting BERT training job...")
    
    if sync:
        print(f"   [WAIT] Running synchronously (will wait for completion)...")
        print(f"   Monitor: https://console.cloud.google.com/vertex-ai/training/custom-jobs")
    else:
        print(f"   [ASYNC] Running asynchronously (will return immediately)")
    
    # Run the job using the SDK method directly
    job.run(sync=sync)
    
    if sync:
        print(f"\n[SUCCESS] BERT training job completed!")
        print(f"   Job name: {job.display_name}")
        print(f"   Job state: {job.state}")
    else:
        print(f"\n[SUCCESS] BERT training job submitted!")
        print(f"   Use job.wait() to wait for completion")
    
    return job

# Hyperparameter Tuning API 
# Create and run hyperparameter tuning jobs.

In [ ]:
def create_hyperparameter_tuning_job(
    config: VertexAIConfig,
    tuning_config: HyperparameterTuningConfig,
    train_data_uri: str,
    val_data_uri: str,
    test_data_uri: str,
    training_script_path: str = "vertex_ai_training.py"
) -> aiplatform.HyperparameterTuningJob:
    """
    Create a hyperparameter tuning job.
    
    Args:
        config: VertexAIConfig instance
        tuning_config: HyperparameterTuningConfig with search space
        train_data_uri: GCS URI to training data
        val_data_uri: GCS URI to validation data
        test_data_uri: GCS URI to test data
        training_script_path: Path to training script
        
    Returns:
        HyperparameterTuningJob instance (not yet started)
    """
    print(f"[INFO] Creating hyperparameter tuning job: {tuning_config.display_name}")
    print(f"   Max trials: {tuning_config.max_trial_count}")
    print(f"   Parallel trials: {tuning_config.parallel_trial_count}")
    
    tuning_job = vai.create_vertex_ai_hyperparameter_tuning_job(
        display_name=tuning_config.display_name,
        training_script_path=training_script_path,
        train_data_gcs_uri=train_data_uri,
        val_data_gcs_uri=val_data_uri,
        test_data_gcs_uri=test_data_uri,
        base_output_dir=f"gs://{config.bucket_name}/hp-tuning",
        max_trial_count=tuning_config.max_trial_count,
        parallel_trial_count=tuning_config.parallel_trial_count,
        project_id=config.project_id,
        location=config.location,
        credentials_path=config.credentials_path,
        metric_name=tuning_config.metric_id,
        metric_goal=tuning_config.metric_goal
    )
    
    print(f"[SUCCESS] Hyperparameter tuning job created: {tuning_config.display_name}")
    return tuning_job

def run_hyperparameter_tuning_job(
    tuning_job: aiplatform.HyperparameterTuningJob,
    sync: bool = False
) -> aiplatform.HyperparameterTuningJob:
    """
    Run a hyperparameter tuning job.
    
    Args:
        tuning_job: HyperparameterTuningJob instance
        sync: If True, wait for completion (not recommended, takes hours)
        
    Returns:
        HyperparameterTuningJob instance
    """
    print(f"[START]  Starting hyperparameter tuning...")
    
    if sync:
        print(f"   [WAIT] Running synchronously (this will take hours)...")
    else:
        print(f"   [ASYNC] Running asynchronously (recommended)")
    
    tuning_job.run(sync=sync)
    
    if not sync:
        print(f"\n[SUCCESS] Hyperparameter tuning job submitted!")
        print(f"   Monitor: https://console.cloud.google.com/vertex-ai/training/hyperparameter-tuning-jobs")
    
    return tuning_job

def get_best_hyperparameters(
    tuning_job: aiplatform.HyperparameterTuningJob
) -> Optional[Dict[str, Any]]:
    """
    Get best trial from completed tuning job.
    
    Args:
        tuning_job: Completed HyperparameterTuningJob
        
    Returns:
        Dictionary with best hyperparameters and metrics
    """
    try:
        trials = tuning_job.trials
        if not trials:
            print("[WARNING]  No trials found")
            return None
        
        # Find best trial
        best_trial = max(trials, key=lambda t: t.final_measurement.metrics[0].value)
        
        result = {
            "trial_id": best_trial.id,
            "parameters": {
                param.parameter_id: param.value 
                for param in best_trial.parameters
            },
            "metrics": {
                metric.metric_id: metric.value
                for metric in best_trial.final_measurement.metrics
            }
        }
        
        print(f"[RESULT] Best Trial: {result['trial_id']}")
        print(f"\n   Parameters:")
        for param, value in result['parameters'].items():
            print(f"      {param}: {value}")
        print(f"\n   Metrics:")
        for metric, value in result['metrics'].items():
            print(f"      {metric}: {value:.4f}")
        
        return result
        
    except Exception as e:
        print(f"[ERROR] Error retrieving best trial: {str(e)}")
        return None